# Statistics for Data Analysts/Engineers — Module 7: Hypothesis Testing Fundamentals

This is the hinge module of the whole course. So far we've estimated unknown quantities
(means, proportions) and reported their uncertainty (confidence intervals). Now we'll
learn to **make decisions** from data: is an observed difference a real effect, or just
random sample noise? That question sounds simple, but it needs a precise conceptual
toolkit, which we'll build in this module -- and use in every subsequent test through the
end of the course.

## Table of contents
1. [The logic of hypothesis testing: H0 and H1](#sec1)
2. [The p-value](#sec2)
3. [Significance level and the decision rule](#sec3)
4. [Type I and Type II errors, statistical power](#sec4)
5. [A full worked example: a one-proportion test](#sec5)
6. [Building intuition for the p-value through simulation](#sec6)
7. [Summary and exercises](#sec7)


Every module in this course starts by generating the same synthetic dataset -- thanks to a fixed random seed (`np.random.default_rng(42)`), the numbers always come out identical, so you can open any module independently of the others, even after restarting the kernel.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
n = 400

city = rng.choice(
    ["New York", "Chicago", "Austin", "Seattle", "Denver"],
    size=n, p=[0.30, 0.25, 0.15, 0.15, 0.15],
)
region = rng.choice(["North", "South", "East", "West"], size=n)
age = rng.integers(18, 70, size=n)

# monthly spend: right-skewed distribution (typical for spending data)
monthly_spend = rng.lognormal(mean=5.2, sigma=0.5, size=n).round(2)

# satisfaction correlated with spend + a region effect (deliberately different) + random
# noise, clipped to a 1-10 scale -- this gives Module 10 (ANOVA) a real between-group gap
region_effect = {"North": 0.0, "South": -0.5, "East": 0.3, "West": 0.8}
satisfaction = np.clip(
    monthly_spend / 45
    + np.array([region_effect[r] for r in region])
    + rng.normal(0, 1.5, size=n),
    1, 10,
).round(1)

# A/B campaign group -- with a REAL injected effect on conversion (group B better)
group = rng.choice(["A", "B"], size=n)
conversion_prob = np.where(group == "A", 0.10, 0.20)
converted = rng.binomial(1, conversion_prob)

# spend before/after a loyalty-program rollout (paired data)
spend_before = rng.normal(220, 40, size=n).round(2)
program_effect = rng.normal(15, 10, size=n)
spend_after = (spend_before + program_effect).round(2)

customers = pd.DataFrame({
    "customer_id": np.arange(1, n + 1),
    "city": city,
    "region": region,
    "age": age,
    "monthly_spend": monthly_spend,
    "satisfaction": satisfaction,
    "campaign_group": group,
    "converted": converted,
    "spend_before": spend_before,
    "spend_after": spend_after,
})
customers.head()


<a id="sec1"></a>
## 1. The logic of hypothesis testing: H0 and H1

Hypothesis testing works a bit like a criminal trial: we start by assuming "innocence"
and look for evidence strong enough to overturn it.

- **Null hypothesis (H0)** — the status quo, "nothing is going on", "there's no
  difference/effect". This is the hypothesis we *default to assuming true* until the data
  provides strong enough evidence against it.
- **Alternative hypothesis (H1 or Ha)** — what we're trying to demonstrate: that there
  IS a difference/effect.

Key point: a statistical test **never "proves" H0**. We can only **reject H0** (when the
evidence is strong enough) or **fail to reject H0** (when there isn't enough evidence) --
we never say "we proved H0" or "H0 is true", exactly like a courtroom never delivers a
verdict of "proven innocent", only "guilty" or "not guilty for lack of sufficient
evidence".

Here's the example we'll carry through this whole module: an e-commerce company claims
that a typical conversion rate in this industry is **12%**. We want to check whether our
shop (the entire `customers` dataset, now treated as a sample from a broader population of
shops) differs from that industry figure.

- H0: `p = 0.12` (our shop is no different from the industry average)
- H1: `p != 0.12` (our shop differs from the industry average — in either direction)

In [ ]:
p_hat = customers["converted"].mean()
n = len(customers)
print(f"Observed conversion rate: {p_hat:.3f} (n={n})")


<a id="sec2"></a>
## 2. The p-value

The **p-value** is the probability of observing a result *at least as extreme* as the one
we actually got, **assuming H0 is true**. In other words: "if the truth really matched
H0, how surprising/unusual would our result be?".

- **A small p-value** (e.g. `0.01`) → our result would be very unlikely if H0 were true →
  strong evidence AGAINST H0.
- **A large p-value** (e.g. `0.60`) → our result is quite ordinary even assuming H0 is
  true → not enough evidence against H0.

> ⚠️ **The most common misinterpretations of the p-value**
>
> The p-value is **NOT**: (1) the probability that H0 is true; (2) the probability that the result happened 'by chance'; (3) a measure of effect size (a small p-value doesn't mean 'a large effect' -- it could be a very small, practically negligible effect detected thanks to a huge sample). The p-value is EXCLUSIVELY a conditional probability: `P(data at least this extreme | H0 is true)`.

<a id="sec3"></a>
## 3. Significance level and the decision rule

The **significance level** (`alpha`, usually `0.05`) is a threshold set BEFORE seeing the
data -- how small `p` has to be for us to consider the evidence "strong enough". The
decision rule is simple:

- if `p <= alpha` → **reject H0** (a "statistically significant" result),
- if `p > alpha` → **fail to reject H0** (not enough evidence to overturn it).

`alpha = 0.05` is a convention (not a law of nature!) -- it means we're willing to accept
a 5% risk of wrongly rejecting a true H0. In higher-stakes contexts (e.g. medical
research), stricter thresholds are often used (`0.01` or lower).

> 📊 **Set alpha in advance, not after the fact**
>
> Setting `alpha` AFTER seeing the p-value (e.g. "it came out 0.07, so let's use alpha=0.10 to make it significant") is a serious methodological error, sometimes called *p-hacking*. The decision threshold must be declared before the analysis, otherwise the whole interpretation loses its meaning.

<a id="sec4"></a>
## 4. Type I and Type II errors, statistical power

A statistical test can be wrong in two different ways:

|                        | H0 is true                     | H0 is false                    |
|------------------------|---------------------------------|----------------------------------|
| **Reject H0**          | Type I error (false alarm)     | Correct decision                 |
| **Fail to reject H0**  | Correct decision               | Type II error (missed effect)    |

- **Type I error** (false positive) — we reject H0 even though it's actually true (e.g.
  we "detect" an effect that isn't there). The probability of this error is exactly
  `alpha`.
- **Type II error** (false negative) — we fail to reject H0 even though it's actually
  false (e.g. we miss a real effect). We denote its probability `beta`.
- **Statistical power** — the probability of CORRECTLY detecting an effect when it
  actually exists: `power = 1 - beta`. We'll return to power and sample-size planning in
  detail in Module 14 (A/B testing).

Lowering `alpha` (to reduce false alarms) automatically **increases** the risk of a
Type II error (it becomes harder to detect anything) -- this is a fundamental trade-off;
you can't improve both at once without increasing sample size.

<a id="sec5"></a>
## 5. A full worked example: a one-proportion test

Back to our question: is `p = 0.12` (H0) consistent with the observed `p_hat`? The test
statistic for a one-proportion test is a z-score (from Module 4):

```
z = (p_hat - p0) / sqrt(p0 * (1 - p0) / n)
```

where `p0` is the value from H0 (here: `0.12`). We compute how many standard deviations
our observed result is from the value H0 assumes, then convert `z` into a p-value using
the normal distribution's CDF.

In [ ]:
from scipy import stats

p0 = 0.12  # the value from H0
se_under_h0 = (p0 * (1 - p0) / n) ** 0.5

z = (p_hat - p0) / se_under_h0

# a two-sided test (H1: p != p0) -- we multiply by 2 since we care about both "tails"
p_value = 2 * (1 - stats.norm.cdf(abs(z)))

print(f"z = {z:.3f}")
print(f"p-value = {p_value:.4f}")

alpha = 0.05
if p_value <= alpha:
    print(f"p <= alpha ({alpha}) -> reject H0")
else:
    print(f"p > alpha ({alpha}) -> fail to reject H0")


`statsmodels` has a ready-made function for exactly this test (`proportions_ztest`),
worth knowing as a shortcut:

In [ ]:
from statsmodels.stats.proportion import proportions_ztest

num_successes = customers["converted"].sum()
z_sm, p_sm = proportions_ztest(count=num_successes, nobs=n, value=p0)
print(f"z = {z_sm:.3f}, p-value = {p_sm:.4f}")


<a id="sec6"></a>
## 6. Building intuition for the p-value through simulation

Instead of computing the p-value from a formula, we can **estimate it via simulation** --
this builds a much better intuition for what the number actually means. We simulate
thousands of "imaginary shops" ASSUMING H0 is true (`p = 0.12`), and check how often a
result as extreme as our `p_hat` shows up by pure chance.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
num_simulations = 20_000

# simulate n Bernoulli trials with p=0.12 (i.e. assuming H0), num_simulations times
simulated_conversions = rng.binomial(n=n, p=p0, size=num_simulations) / n

# how often is the simulated result AS OR MORE extreme (in either direction) than our p_hat?
observed_deviation = abs(p_hat - p0)
simulated_p_value = (np.abs(simulated_conversions - p0) >= observed_deviation).mean()

print(f"P-value from simulation: {simulated_p_value:.4f}")
print(f"P-value from the formula (section 5): {p_value:.4f}")

plt.figure(figsize=(8, 4))
plt.hist(simulated_conversions, bins=50, color="lightgray", edgecolor="white")
plt.axvline(p_hat, color="red", linestyle="--", label=f"Our observed result: {p_hat:.3f}")
plt.axvline(p0, color="black", linestyle="-", label=f"H0: p = {p0}")
plt.xlabel("Simulated conversion rate (assuming H0)")
plt.title("Distribution of outcomes under H0 -- where does our result fall?")
plt.legend()
plt.show()


The p-value from the formula and from the simulation should come out very close --
because it's exactly the same question, computed by two different methods. You can also
see it directly on the chart: the red line (our result) sits clearly at the edge of the
"cloud" of possible outcomes under H0 -- that's exactly what a small p-value means.

<a id="sec7"></a>
## 7. Summary and exercises

In this module we covered:
- the H0/H1 logic (the "innocence" assumption we try to overturn with evidence),
- what a p-value really is (and isn't),
- the significance level `alpha` and the decision rule,
- Type I and Type II errors and statistical power,
- a full, hand-computed test for a single proportion (a z-test),
- the simulation-based intuition behind the p-value.

> 📝 **Exercise 1: A test against a different benchmark**
>
> Repeat the test from section 5, but for `p0 = 0.15` instead of `0.12`. Do we still reject H0 at the `alpha=0.05` level with this benchmark?

<details>
<summary><b>👉 Click to reveal a sample solution</b></summary>

```python
p0_b = 0.15
se_b = (p0_b * (1 - p0_b) / n) ** 0.5
z_b = (p_hat - p0_b) / se_b
p_value_b = 2 * (1 - stats.norm.cdf(abs(z_b)))
print(f"z={z_b:.3f}, p={p_value_b:.4f}")
print("Reject H0" if p_value_b <= 0.05 else "Fail to reject H0")
```
</details>

> 📝 **Exercise 2: A one-sided test**
>
> The company claims the TYPICAL conversion rate is AT LEAST 12% (so H1 is now one-sided: `p < 0.12`, testing whether we're BELOW the benchmark). Change the formula from section 5 to a one-sided test: the p-value is now simply `stats.norm.cdf(z)` (no multiplying by 2, and no `abs()`). Compute it and compare with the two-sided test result.

<details>
<summary><b>👉 Click to reveal a sample solution</b></summary>

```python
one_sided_p_value = stats.norm.cdf(z)
print(f"z={z:.3f}, one-sided p={one_sided_p_value:.4f}")
print(f"(for comparison, two-sided p from section 5: {p_value:.4f})")
```

A one-sided test gives a smaller p-value for the same z (since the whole 'budget' of alpha significance goes into one tail) -- but it can ONLY be used when the direction of the effect was decided BEFORE seeing the data, otherwise it's another form of p-hacking.
</details>

> 📝 **Exercise 3: Simulation for a different sample size**
>
> Repeat the simulation from section 6, but for `n=50` instead of `n=400` (with the same observed `p_hat=0.185`, rounded to the nearest integer number of successes). Is the same observed rate still statistically significant with a smaller sample?

<details>
<summary><b>👉 Click to reveal a sample solution</b></summary>

```python
small_n = 50
small_p_hat = 0.185  # same observed rate, smaller sample
small_successes = round(small_p_hat * small_n)

small_simulated = rng.binomial(n=small_n, p=p0, size=20_000) / small_n
small_deviation = abs(small_successes / small_n - p0)
small_p_value = (np.abs(small_simulated - p0) >= small_deviation).mean()
print(f"P-value at n={small_n}: {small_p_value:.4f}")
```

With a smaller sample, the same observed rate usually gives a LARGER p-value (less evidence) -- a direct illustration of why sample size matters (Module 5 and Module 14).
</details>

> 🔥 **Challenge: a power curve**
>
> Write a simulation to check the POWER of the test (the probability of correctly rejecting H0 when the TRUE `p` is actually `0.16`, different from H0's `p0=0.12`). For `n=100`: simulate 2000 'shops' with a true `p=0.16`, compute the test's p-value for each one (H0: `p0=0.12`), and compute the fraction of cases where `p <= 0.05` (i.e. the test correctly detected the difference). That fraction is the approximate power of the test for this specific effect and this sample size.

<details>
<summary><b>👉 Click to reveal a sample solution</b></summary>

```python
true_p = 0.16
power_n = 100
num_shops = 2000
detections = 0

for _ in range(num_shops):
    sim_successes = rng.binomial(n=power_n, p=true_p)
    sim_p_hat = sim_successes / power_n
    sim_se = (p0 * (1 - p0) / power_n) ** 0.5
    sim_z = (sim_p_hat - p0) / sim_se
    sim_p_value = 2 * (1 - stats.norm.cdf(abs(sim_z)))
    if sim_p_value <= 0.05:
        detections += 1

print(f"Approximate power of the test: {detections / num_shops:.3f}")
```

The result shows how often the test CORRECTLY detects a real difference of this size at n=100 -- power below about 0.80 is usually considered too low in practice (more on this in Module 14).
</details>

## What's next?

We now have the full conceptual toolkit: H0/H1, the p-value, alpha, Type I/II errors. In
**Module 8** we'll apply it to the most commonly used family of tests -- **t-tests** -- for
comparing means: a single sample against a reference value, two independent groups, and
paired data (like our `spend_before`/`spend_after`).